# Neural Network From Scratch for Telugu Laghu/Guru Data

This notebook trains a small neural network using only NumPy. The dataset contains Telugu words and their laghu/guru pattern, where `I` and `U` are the target symbols.

Because the file has only a word-level pattern label, not a syllable-by-syllable alignment, the model learns this task as:

```text
Telugu word -> short sequence of I/U labels
```

This is useful as a learning notebook for backpropagation and sequence-style output. With only 67 rows, it will overfit easily and should not be treated as a production chandassu analyzer.


## Load the Dataset

Each non-empty line in `lagu_guru_data.txt` is expected to contain a word followed by its laghu/guru pattern.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np

DATA_PATH = Path("lagu_guru_data.txt")


def load_lagu_guru_data(path):
    rows = []

    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        parts = line.strip().split()
        if not parts:
            continue
        if len(parts) < 2:
            raise ValueError(f"Line {line_no} must contain a word and a pattern: {line!r}")

        word = " ".join(parts[:-1])
        pattern = parts[-1].upper()

        if not set(pattern).issubset({"I", "U"}):
            raise ValueError(f"Line {line_no} has invalid pattern {pattern!r}")

        rows.append((word, pattern))

    return rows


rows = load_lagu_guru_data(DATA_PATH)

print(f"Samples: {len(rows)}")
print(f"Unique patterns: {len(set(pattern for _, pattern in rows))}")
print(f"Most common patterns: {Counter(pattern for _, pattern in rows).most_common(8)}")
rows[:10]


## Encode Words and Pattern Sequences

The input is a fixed-length one-hot encoding of Telugu characters. The target is a fixed-length sequence where:

- `0` means padding after the pattern ends
- `1` means `I`
- `2` means `U`

For example, `UI` becomes `[2, 1, 0, 0, ...]`.


In [ ]:
characters = sorted(set("".join(word for word, _ in rows)))
char_to_idx = {char: idx for idx, char in enumerate(characters)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

symbol_to_idx = {"PAD": 0, "I": 1, "U": 2}
idx_to_symbol = {0: "", 1: "I", 2: "U"}

max_word_length = max(len(word) for word, _ in rows)
max_pattern_length = max(len(pattern) for _, pattern in rows)
vocab_size = len(characters)
n_output_symbols = len(symbol_to_idx)


def vectorize_word(word):
    x = np.zeros((max_word_length, vocab_size), dtype=np.float32)

    for position, char in enumerate(word[:max_word_length]):
        if char in char_to_idx:
            x[position, char_to_idx[char]] = 1.0

    return x.reshape(-1)


def encode_pattern(pattern):
    y = np.zeros(max_pattern_length, dtype=np.int64)

    for position, symbol in enumerate(pattern[:max_pattern_length]):
        y[position] = symbol_to_idx[symbol]

    return y


def decode_pattern(encoded_pattern):
    return "".join(idx_to_symbol[int(value)] for value in encoded_pattern if int(value) != 0)


X = np.array([vectorize_word(word) for word, _ in rows], dtype=np.float32)
y = np.array([encode_pattern(pattern) for _, pattern in rows], dtype=np.int64)

print(f"Character vocabulary size: {vocab_size}")
print(f"Max word length: {max_word_length}")
print(f"Max pattern length: {max_pattern_length}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


## Train/Test Split

The split keeps one-sample pattern classes in the training set. This avoids testing on a class pattern that the model never had a chance to see.


In [ ]:
rng = np.random.default_rng(7)
indices_by_pattern = defaultdict(list)

for idx, (_, pattern) in enumerate(rows):
    indices_by_pattern[pattern].append(idx)

train_indices = []
test_indices = []

for pattern, pattern_indices in indices_by_pattern.items():
    pattern_indices = np.array(pattern_indices)
    rng.shuffle(pattern_indices)

    if len(pattern_indices) == 1:
        train_indices.extend(pattern_indices.tolist())
    else:
        test_count = max(1, int(round(0.2 * len(pattern_indices))))
        test_indices.extend(pattern_indices[:test_count].tolist())
        train_indices.extend(pattern_indices[test_count:].tolist())

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)
rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train, y_train = X[train_indices], y[train_indices]
X_test, y_test = X[test_indices], y[test_indices]

print(f"Train samples: {len(train_indices)}")
print(f"Test samples: {len(test_indices)}")


## Neural Network Layers From Scratch

The model is a small MLP:

```text
one-hot Telugu word -> Dense -> ReLU -> Dense -> pattern-symbol logits
```

The final Dense layer emits `max_pattern_length * 3` values, reshaped into one 3-class prediction for each pattern position.


In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons, weight_regularizer_l2=0.0):
        self.weights = 0.01 * rng.standard_normal((n_inputs, n_neurons))
        self.biases = np.zeros((1, n_neurons))
        self.weight_regularizer_l2 = weight_regularizer_l2

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)

        if self.weight_regularizer_l2 > 0:
            self.dweights += 2 * self.weight_regularizer_l2 * self.weights

        self.dinputs = np.dot(dvalues, self.weights.T)

    def regularization_loss(self):
        if self.weight_regularizer_l2 == 0:
            return 0.0
        return self.weight_regularizer_l2 * np.sum(self.weights * self.weights)


class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Sequence_Softmax_Crossentropy:
    def forward(self, logits, y_true):
        samples = logits.shape[0]
        logits = logits.reshape(samples, max_pattern_length, n_output_symbols)
        shifted_logits = logits - np.max(logits, axis=2, keepdims=True)
        exp_values = np.exp(shifted_logits)
        self.output = exp_values / np.sum(exp_values, axis=2, keepdims=True)
        self.y_true = y_true

        clipped = np.clip(self.output, 1e-7, 1 - 1e-7)
        correct_confidences = clipped[
            np.arange(samples)[:, None],
            np.arange(max_pattern_length)[None, :],
            y_true,
        ]

        return np.mean(-np.log(correct_confidences))

    def backward(self):
        samples = self.output.shape[0]
        dinputs = self.output.copy()
        dinputs[
            np.arange(samples)[:, None],
            np.arange(max_pattern_length)[None, :],
            self.y_true,
        ] -= 1

        dinputs /= samples * max_pattern_length
        self.dinputs = dinputs.reshape(samples, max_pattern_length * n_output_symbols)


class Optimizer_Adam:
    def __init__(self, learning_rate=0.03, decay=1e-4, epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, "weight_cache"):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        weight_momentums_corrected = layer.weight_momentums / (1 - self.beta_1 ** (self.iterations + 1))
        bias_momentums_corrected = layer.bias_momentums / (1 - self.beta_1 ** (self.iterations + 1))

        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * layer.dweights**2
        layer.bias_cache = self.beta_2 * layer.bias_cache + (1 - self.beta_2) * layer.dbiases**2

        weight_cache_corrected = layer.weight_cache / (1 - self.beta_2 ** (self.iterations + 1))
        bias_cache_corrected = layer.bias_cache / (1 - self.beta_2 ** (self.iterations + 1))

        layer.weights += -self.current_learning_rate * weight_momentums_corrected / (
            np.sqrt(weight_cache_corrected) + self.epsilon
        )
        layer.biases += -self.current_learning_rate * bias_momentums_corrected / (
            np.sqrt(bias_cache_corrected) + self.epsilon
        )

    def post_update_params(self):
        self.iterations += 1


## Build and Train the Model


In [ ]:
dense1 = Layer_Dense(X_train.shape[1], 96, weight_regularizer_l2=1e-5)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(96, max_pattern_length * n_output_symbols, weight_regularizer_l2=1e-5)
loss_activation = Sequence_Softmax_Crossentropy()
optimizer = Optimizer_Adam(learning_rate=0.03, decay=1e-4)


def predict_proba(X_values):
    dense1.forward(X_values)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    logits = dense2.output.reshape(len(X_values), max_pattern_length, n_output_symbols)
    shifted_logits = logits - np.max(logits, axis=2, keepdims=True)
    exp_values = np.exp(shifted_logits)
    return exp_values / np.sum(exp_values, axis=2, keepdims=True)


def accuracy_scores(X_values, y_values):
    predictions = np.argmax(predict_proba(X_values), axis=2)
    sequence_accuracy = np.mean(np.all(predictions == y_values, axis=1))
    symbol_accuracy = np.mean(predictions == y_values)
    return sequence_accuracy, symbol_accuracy


for epoch in range(3001):
    dense1.forward(X_train)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    data_loss = loss_activation.forward(dense2.output, y_train)
    regularization_loss = dense1.regularization_loss() + dense2.regularization_loss()
    loss = data_loss + regularization_loss

    if epoch % 500 == 0:
        sequence_accuracy, symbol_accuracy = accuracy_scores(X_train, y_train)
        print(
            f"epoch: {epoch:4d}, "
            f"seq_acc: {sequence_accuracy:.3f}, "
            f"symbol_acc: {symbol_accuracy:.3f}, "
            f"loss: {loss:.4f}, "
            f"lr: {optimizer.current_learning_rate:.5f}"
        )

    loss_activation.backward()
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()


## Evaluate the Model

Sequence accuracy means the entire predicted pattern must match. Symbol accuracy checks each `I`, `U`, or padding position separately.


In [ ]:
train_sequence_accuracy, train_symbol_accuracy = accuracy_scores(X_train, y_train)
test_sequence_accuracy, test_symbol_accuracy = accuracy_scores(X_test, y_test)

print(f"Train sequence accuracy: {train_sequence_accuracy:.3f}")
print(f"Train symbol accuracy:   {train_symbol_accuracy:.3f}")
print(f"Test sequence accuracy:  {test_sequence_accuracy:.3f}")
print(f"Test symbol accuracy:    {test_symbol_accuracy:.3f}")


## Inspect Test Predictions


In [ ]:
test_probabilities = predict_proba(X_test)
test_predictions = np.argmax(test_probabilities, axis=2)

for original_idx, predicted_encoded in zip(test_indices, test_predictions):
    word, expected_pattern = rows[original_idx]
    predicted_pattern = decode_pattern(predicted_encoded)
    mark = "OK" if predicted_pattern == expected_pattern else "MISS"
    print(f"{mark:4s}  {word:16s} expected={expected_pattern:8s} predicted={predicted_pattern}")


## Predict a Pattern for a Word

This helper uses the trained model above. For words outside the small dataset, treat predictions as experimental.


In [ ]:
def predict_pattern(word, top_k=3):
    x = vectorize_word(word).reshape(1, -1)
    probabilities = predict_proba(x)[0]
    best_symbols = np.argmax(probabilities, axis=1)
    predicted_pattern = decode_pattern(best_symbols)
    confidence = np.mean(np.max(probabilities, axis=1))
    return predicted_pattern, float(confidence)


for word in ["వేమ", "మంచివాని", "గురువా", "తెలిసి"]:
    pattern, confidence = predict_pattern(word)
    print(f"{word:12s} -> {pattern:8s} confidence={confidence:.3f}")


## Optional: Train on All Rows for a Memorization Demo

The train/test split above is better for checking behavior. Since this dataset is tiny, the model can also be trained on all rows when you only want to see whether the network can learn the provided examples.


In [ ]:
dense1 = Layer_Dense(X.shape[1], 96, weight_regularizer_l2=1e-5)
activation1 = Activation_ReLU()
dense2 = Layer_Dense(96, max_pattern_length * n_output_symbols, weight_regularizer_l2=1e-5)
loss_activation = Sequence_Softmax_Crossentropy()
optimizer = Optimizer_Adam(learning_rate=0.03, decay=1e-4)

for epoch in range(2501):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)

    data_loss = loss_activation.forward(dense2.output, y)
    regularization_loss = dense1.regularization_loss() + dense2.regularization_loss()
    loss = data_loss + regularization_loss

    if epoch % 500 == 0:
        sequence_accuracy, symbol_accuracy = accuracy_scores(X, y)
        print(
            f"epoch: {epoch:4d}, "
            f"seq_acc: {sequence_accuracy:.3f}, "
            f"symbol_acc: {symbol_accuracy:.3f}, "
            f"loss: {loss:.4f}, "
            f"lr: {optimizer.current_learning_rate:.5f}"
        )

    loss_activation.backward()
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_update_params()

full_sequence_accuracy, full_symbol_accuracy = accuracy_scores(X, y)
print(f"Full-data sequence accuracy: {full_sequence_accuracy:.3f}")
print(f"Full-data symbol accuracy:   {full_symbol_accuracy:.3f}")

full_predictions = np.argmax(predict_proba(X), axis=2)
for (word, expected_pattern), predicted_encoded in zip(rows[:12], full_predictions[:12]):
    print(f"{word:16s} expected={expected_pattern:8s} predicted={decode_pattern(predicted_encoded)}")
